In [1]:
# ─────────────────────────────────────────────
# [C1] ◀ ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))
bikes = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
    "bike_sharing.zip", "day.csv"))

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"자전거 대여 데이터: {bikes.shape[0]:,}행 × {bikes.shape[1]}열")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
내려받는 중… bike_sharing.zip
쇼핑 세션 데이터: 12,330행 × 18열
자전거 대여 데이터: 731행 × 16열

→ 준비 완료. 이제 여러분 차례입니다.


In [17]:
from IPython.display import display

# 1) 데이터 미리보기
print("=== head(5) ===")
display(shoppers.head(5))

# 2) 데이터 구조 정보
print("\n=== info() ===")
shoppers.info()

# 3) Revenue 클래스 분포 (개수 + 비율을 하나의 표로)
revenue_summary = pd.DataFrame({
    "count": shoppers["Revenue"].value_counts(),
    "ratio": shoppers["Revenue"].value_counts(normalize=True).round(4)
})
print("\n=== Revenue 클래스 분포 ===")
display(revenue_summary)

=== head(5) ===


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False



=== info() ===
<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType      

,count,ratio
Revenue,,
False,10422,0.8453
True,1908,0.1547


1. Revenue는 true/false이기 때문에 분류 문제에 속한다

In [8]:
# [C3] ◀ 문제 2. 분류 — 베이스라인부터 세운다
# ⌨️ 문제 2 — 베이스라인(Dummy) 정확도
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

# 여기에 코드를 작성하세요 (X, y 준비 → 분리 → Dummy 학습 → 테스트 정확도)
X = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_score = dummy.score(X_test, y_test)

print(f"DummyClassifier 정확도: {dummy_score:.4f} (무작위 추측 기준)")

DummyClassifier 정확도: 0.8451 (무작위 추측 기준)


In [12]:
# [C4] ◀ 문제 3. 분류 — 로지스틱 회귀로 베이스라인을 넘어라
# ⌨️ 문제 3 — 로지스틱 회귀 학습·평가 (베이스라인과 비교)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 여기에 코드를 작성하세요 (스케일링 → 학습 → 학습/테스트 정확도 비교)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # 학습 데이터로만 기준을 잡고
X_test_s = scaler.transform(X_test) 


clf = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)              # 예/아니오 예측
y_proba = clf.predict_proba(X_test_s)[:, 1] # 1(이탈)일 확률

train_score = clf.score(X_train_s, y_train)
test_score = clf.score(X_test_s, y_test)
print(f"학습 정확도: {train_score:.4f}")
print(f"테스트 정확도: {test_score:.4f}")
print(f"DummyClassifier 정확도: {dummy_score:.4f}")

학습 정확도: 0.8839
테스트 정확도: 0.8796
DummyClassifier 정확도: 0.8451


1. 과적합 신호 = 없음
  - 일반화 격차 = 0.8839−0.8796=0.0043, 이정도 격차는 과적합이라고 부르기에는 적은 양이다

2. 로지스틱 회귀가 더미 모델보다 약 3.5%p 높다
3. 테스트가 학습 정확도보다 근소하게 낮음

In [18]:
# [C5] ◀ 문제 4. 회귀 — 하루 대여량을 예측하라
# ⌨️ 문제 4 — 자전거 대여량 회귀 (Dummy → LinearRegression)
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

FEATURES = ["temp", "atemp", "hum", "windspeed"]

# 여기에 코드를 작성하세요 (X, y → 분리 → Dummy R² vs 선형회귀 R²)
X = bikes[FEATURES]
y = bikes["cnt"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
linear = LinearRegression()
linear.fit(X_train, y_train)
yr_pred = linear.predict(X_test)

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

r2 = r2_score(y_test, yr_pred)
train_r2 = linear.score(X_train, y_train)

print(f"학습 R² = {train_r2:.3f}")
print(f"테스트 R²  = {r2:.3f}")
print(f"Dummy R²  = {r2_score(y_test, dummy_pred):.3f}")
print(f"→ 이 모델은 다음 달 시청 시간 변동의 약 {r2*100:.0f}%를 설명합니다.")

학습 R² = 0.450
테스트 R²  = 0.499
Dummy R²  = -0.020
→ 이 모델은 다음 달 시청 시간 변동의 약 50%를 설명합니다.


In [19]:
from sklearn.metrics import r2_score

train_r2 = linear.score(X_train, y_train)
test_r2 = r2_score(y_test, yr_pred)   # 이미 있는 r2 변수와 동일 (r2_score(y_test, yr_pred))

gap = train_r2 - test_r2

print(f"학습 R² = {train_r2:.3f}")
print(f"테스트 R² = {test_r2:.3f}")
print(f"격차(학습 - 테스트) = {gap:.3f}")

if gap > 0.1:
    print("→ 과적합 신호 있음: 학습 데이터에만 잘 맞고 있을 가능성 (10%p 이상 격차)")
elif gap > 0.05:
    print("→ 약한 과적합 신호: 격차가 다소 있음 (5~10%p), 주의 깊게 볼 필요")
else:
    print("→ 과적합 신호 거의 없음: 학습/테스트 성능이 비슷한 수준")

학습 R² = 0.450
테스트 R² = 0.499
격차(학습 - 테스트) = -0.049
→ 과적합 신호 거의 없음: 학습/테스트 성능이 비슷한 수준
